## Install Required Dependecies

In [1]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.0 MB/s eta 0:00:00


In [2]:
!pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Necessary Imports

In [3]:
# Necessary Imports

import os
from openai import OpenAI
from google.colab import userdata

from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


from abc import ABC, abstractmethod
from typing import Any, List, Tuple

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.documents import Document

In [4]:
# Load OPENAI API KEY

api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## Intent Classification Layer

In [6]:
class IntentCategory(str, Enum):
    FACTUAL_FAQ = "FACTUAL_FAQ"
    PROCEDURAL_GUIDANCE = "PROCEDURAL_GUIDANCE"
    COMMUNITY_EXPERIENCE = "COMMUNITY_EXPERIENCE"
    EVIDENCE_BASED = "EVIDENCE_BASED"
    COMPLEX_MULTI_HOP = "COMPLEX_MULTI_HOP"


class IntentOutput(BaseModel):
    intent: IntentCategory = Field(
        description="The single most appropriate intent category for the user query"
    )


INTENT_CLASSIFIER_SYSTEM_PROMPT = """
You are an intent classification engine for a COVID-19 question-answering system.

Your task:
Given a user query, classify it into EXACTLY ONE intent category.

You must always return one category, even if the query does not clearly match any rules.
Use semantic understanding, not just keywords.

--------------------
INTENT CATEGORIES
--------------------

FACTUAL_FAQ:
- Short, direct factual questions
- Definitions, symptoms, transmission, vaccines, timelines
- Example: "What are the symptoms of COVID-19?"

PROCEDURAL_GUIDANCE:
- Questions asking for steps, actions, prevention, or advice
- Often includes "how", "what should I do", "can I", but do NOT rely only on keywords
- Example: "How can I protect myself from COVID-19?"

COMMUNITY_EXPERIENCE:
- Informal, opinionated, rumor-based, or social media influenced queries
- Mentions of people, beliefs, claims, or uncertainty
- Example: "People say masks don't work, is that true?"

EVIDENCE_BASED:
- Research-oriented, scientific, or evidence-seeking questions
- Mentions studies, data, research, clinical findings, or comparisons
- Example: "What does research say about COVID survival on surfaces?"

COMPLEX_MULTI_HOP:
- Long, multi-part, ambiguous, or mixed-intent questions
- Requires synthesising multiple facts or evidence
- Use this ONLY when no single intent clearly dominates
- Example: "How does COVID affect elderly people and what precautions should families take?"

--------------------
DECISION RULES (GUIDANCE, NOT HARD RULES)
--------------------

1. Use semantic meaning first, keywords second.
2. If multiple intents seem present:
   - Choose the MOST dominant intent.
3. If the query is long, compound, or unclear:
   - Prefer COMPLEX_MULTI_HOP.
4. If safety or scientific grounding is implied:
   - Prefer EVIDENCE_BASED over FACTUAL_FAQ.
5. Never return multiple categories.
6. Never invent new categories.
7. Never explain your reasoning.

Return ONLY the intent category in structured form.
"""

intent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

intent_classifier = intent_llm.with_structured_output(IntentOutput)

def classify_intent_llm(query: str) -> IntentCategory:
    result = intent_classifier.invoke(
        [
            {"role": "system", "content": INTENT_CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ]
    )
    return result.intent.value

## Query Expansion Layer

In [7]:
class QueryExpander:
    """
    LLM-based query expansion for multi-query retrieval.
    """

    DELIMITER = "#next-question#"

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        n_variations: int = 3,
        temperature: float = 0.3,
    ):
        self.n_variations = n_variations
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "n", "delimiter"],
            template="""
You are an expert search query rewriter for retrieval-augmented generation systems.

Rewrite the following user query into {n} alternative versions.
Each version should reflect a different perspective or phrasing
while preserving the original intent.

Separate each rewritten query using the token:
{delimiter}

User query:
"{query}"
""",
        )

    def expand(self, query: str) -> List[str]:
        formatted_prompt = self.prompt.format(
            query=query,
            n=self.n_variations,
            delimiter=self.DELIMITER,
        )

        response = self.llm.invoke(formatted_prompt).content

        expanded_queries = [
            q.strip()
            for q in response.split(self.DELIMITER)
            if q.strip()
        ]

        # Always include original query
        return [query] + expanded_queries


## Multi FAISS Retriever Function

In [8]:
class MultiFAISSRetriever:
    """
    Concurrent retriever supporting:
    - multiple FAISS indexes
    - multiple expanded queries
    - optional similarity scores
    """

    def __init__(
        self,
        vectorstores: List[FAISS],
        top_k: int = 5,
        max_workers: int = 8,
        use_scores: bool = False,
    ):
        self.vectorstores = vectorstores
        self.top_k = top_k
        self.max_workers = max_workers
        self.use_scores = use_scores

    def _search(
        self, vs: FAISS, query: str
    ) -> List[Tuple[Document, float]] | List[Document]:
        """
        Internal helper to perform either:
        - similarity_search
        - similarity_search_with_score
        """
        if self.use_scores:
            return vs.similarity_search_with_score(query, k=self.top_k)
        return vs.similarity_search(query, k=self.top_k)

    def retrieve(
        self, queries: List[str]
    ) -> List[Document] | List[Tuple[Document, float]]:
        """
        Executes concurrent retrieval across all queries and indexes.
        Deduplicates documents across sources.
        """

        seen_doc_ids = set()
        results = []

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [
                executor.submit(self._search, vs, q)
                for vs in self.vectorstores
                for q in queries
            ]

            for future in as_completed(futures):
                retrieved = future.result()

                for item in retrieved:
                    if self.use_scores:
                        doc, score = item
                    else:
                        doc = item
                        score = None

                    # Robust deduplication key
                    doc_id = (
                        doc.metadata.get("id")
                        if doc.metadata and "id" in doc.metadata
                        else hash(doc.page_content)
                    )

                    if doc_id in seen_doc_ids:
                        continue

                    seen_doc_ids.add(doc_id)

                    if self.use_scores:
                        results.append((doc, score))
                    else:
                        results.append(doc)

        return results


## Context Retrieval Layer

In [9]:
# Load Multiple FAISS Vector Stores from Google Drive

def load_faiss_vectorstores(
    vectorstore_paths: List[str],
    embedding_model: OpenAIEmbeddings,
) -> List[FAISS]:
    """
    Loads multiple FAISS vector stores from disk.
    """
    vectorstores = []

    for path in vectorstore_paths:
        vs = FAISS.load_local(
            folder_path=path,
            embeddings=embedding_model,
            allow_dangerous_deserialization=True,  # required for FAISS
        )
        vectorstores.append(vs)

    return vectorstores

# Vector Stores Path

vectorstore_paths = {
  "source_1": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-1-3072-Kaggle-COVID-19-FAQ",

  "source_2": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-2-3072-COVID-QA-community",

  "source_3": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-3-3072-COUGH-FAQ-ENG",

  "source_4": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-4-3072-COVID-QA-MASTER"

}

# Intent Routing

INTENT_ROUTING = {
    "FACTUAL_FAQ": ["source_1", "source_3"],
    "PROCEDURAL_GUIDANCE": ["source_1", "source_3"],
    "COMMUNITY_EXPERIENCE": ["source_2"],
    "EVIDENCE_BASED": ["source_4"],
    "COMPLEX_MULTI_HOP": ["source_1", "source_3", "source_4"],
}

In [10]:
user_query = "What are the common symptoms of COVID-19?"

In [11]:
# Indentify the intent and load respective vector stores
intent = classify_intent_llm(user_query)

sources = INTENT_ROUTING[intent]
stores_paths = [vectorstore_paths[source] for source in sources]

indexes = load_faiss_vectorstores(
    vectorstore_paths=stores_paths,
    embedding_model=embeddings,
)


# # Expand query
expander = QueryExpander(n_variations=3)
expanded_queries = expander.expand(user_query)

In [12]:
# Retrieve documents
retriever = MultiFAISSRetriever(
    vectorstores=indexes,
    top_k=5,
    use_scores=True
)

retrieved_docs_with_scores = retriever.retrieve(expanded_queries)

## Post-Retrieval Layer

### Reranking (Cross-Encoders)

The most prominent post-retrieval technique discussed is reranking, which addresses the limitations of standard vector searches.

- The Problem: Initial searches (bi-encoders) rely on distance-based similarity between query and document embeddings, which may occasionally retrieve documents that are semantically close but contextually unaligned with the user's true intent.

- The Solution: A cross-encoder model is used to score each retrieved chunk against the original user query. Unlike bi-encoders, cross-encoders can identify more complex and nuanced relationships between the question and the text.

- The Workflow: The system retrieves a broader pool of candidates—typically N×K chunks if query expansion was used—and then the reranker reorders them, allowing the system to keep only the absolute best top K results for the final prompt.

In [13]:
from sentence_transformers import CrossEncoder

In [14]:
class CrossEncoderReranker:
    """
    Cross-encoder reranker for post-retrieval optimisation.
    """

    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k: int = 5,
        batch_size: int = 16,
    ):
        self.model = CrossEncoder(model_name)
        self.top_k = top_k
        self.batch_size = batch_size

    def rerank(
        self,
        query: str,
        documents: List[Document],
    ) -> List[Tuple[Document, float]]:
        """
        Reranks retrieved documents using cross-encoder scores.
        """

        pairs = [(query, doc.page_content) for doc in documents]

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
        )

        reranked = list(zip(documents, scores))
        reranked.sort(key=lambda x: x[1], reverse=True)

        return reranked[: self.top_k]


In [15]:
rerank_top_k: int = 5

In [16]:
retrieved_docs = [doc for doc, _ in retrieved_docs_with_scores]

In [17]:
reranker = CrossEncoderReranker(top_k=rerank_top_k)
reranked_docs_with_scores = reranker.rerank(
        query=user_query,
        documents=retrieved_docs,
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [18]:
reranked_docs = [doc for doc, _ in reranked_docs_with_scores]

In [19]:
reranked_docs

[Document(id='9f3910b0-3166-426e-9c7a-a52717843b39', metadata={'dataset': 'COVID19_FAQ', 'original_question': '2. What are the symptoms of COVID-19?', 'record_id': 1, 'chunk_id': 0, 'chunk_type': 'full', 'source': 'Kaggle COVID-19 FAQ', 'embedding_tokens': 166}, page_content="Question: 2. What are the symptoms of COVID-19?\n\nAnswer:\nThe most common symptoms of COVID-19 are fever, tiredness, and dry cough. Some patients   may have aches and pains, nasal congestion, runny nose, sore throat or diarrhea. These   symptoms are usually mild and begin gradually. Some people become infected but dont   develop any symptoms and don't feel unwell. Most people (about 80%) recover from the   disease without needing special treatment. Around 1 out of every 6 people who gets COVID-19   becomes seriously ill and develops difficulty breathing. Older people, and those with underlying   medical problems like high blood pressure, heart problems or diabetes, are more likely to   develop serious illness. P

### Prompt Compression (Context Refinement Layer)

Why Prompt Compression?

Even after reranking:
- Chunks may be verbose
- Redundant details increase cost
- Long prompts → attention bias + hallucination

Goal:
- Preserve meaning, remove noise.

In [20]:
class PromptCompressor:
    """
    Compresses retrieved documents while preserving factual content.
    """

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        temperature: float = 0.0,
        max_tokens: int = 256,
    ):
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="""
You are a system that compresses retrieved context for a question-answering system.

Given the user query and the retrieved text:
- Remove irrelevant or redundant information
- Preserve all medically important facts
- Do NOT add new information
- Keep the output concise and factual

User query:
{query}

Retrieved context:
{context}

Compressed context:
""",
        )

        self.max_tokens = max_tokens

    def compress(
        self,
        query: str,
        documents: List[Document],
    ) -> List[str]:
        compressed_chunks = []

        for doc in documents:
            formatted_prompt = self.prompt.format(
                query=query,
                context=doc.page_content,
            )

            response = self.llm.invoke(
                formatted_prompt,
                max_tokens=self.max_tokens,
            )

            compressed_chunks.append(response.content.strip())

        return compressed_chunks


In [21]:
compressor = PromptCompressor()
compressed_context = compressor.compress(
    query=user_query,
    documents=reranked_docs
    )

In [26]:
compressed_context[0]

'The most common symptoms of COVID-19 are fever, tiredness, and dry cough. Other symptoms may include aches, nasal congestion, runny nose, sore throat, or diarrhea. Symptoms are usually mild and begin gradually. Some people may be infected without developing symptoms. About 80% recover without special treatment, but around 1 in 6 may become seriously ill with difficulty breathing, especially older adults and those with underlying health conditions. People with fever, cough, and difficulty breathing should seek medical attention.'

In [27]:
compressed_context[1]

'The most common symptoms of COVID-19 are fever, tiredness, and dry cough. Other symptoms may include aches and pains, nasal congestion, runny nose, sore throat, or diarrhea. Symptoms are usually mild and begin gradually. Some people may be infected without developing symptoms.'

In [28]:
compressed_context[2]

'The most common symptoms of COVID-19 are fever, dry cough, and tiredness. Less common symptoms include aches and pains, nasal congestion, headache, conjunctivitis, sore throat, diarrhea, loss of taste or smell, rash on skin, or discoloration of fingers.'

In [29]:
compressed_context[3]

'Common symptoms of COVID-19 include fever, cough, shortness of breath, chills, fatigue, muscle or body aches, headache, new loss of taste or smell, sore throat, congestion or runny nose, abdominal pain/discomfort, nausea, vomiting, and diarrhea.'

In [30]:
compressed_context[4]

'The most common symptoms of COVID-19 include fever, cough, and trouble breathing. Other symptoms may include sore throat, congestion, runny nose, chills, muscle pain, headache, loss of taste or smell, nausea or vomiting, diarrhea, and tiredness.'

In [34]:
reranked_docs[0]

Document(id='9f3910b0-3166-426e-9c7a-a52717843b39', metadata={'dataset': 'COVID19_FAQ', 'original_question': '2. What are the symptoms of COVID-19?', 'record_id': 1, 'chunk_id': 0, 'chunk_type': 'full', 'source': 'Kaggle COVID-19 FAQ', 'embedding_tokens': 166}, page_content="Question: 2. What are the symptoms of COVID-19?\n\nAnswer:\nThe most common symptoms of COVID-19 are fever, tiredness, and dry cough. Some patients   may have aches and pains, nasal congestion, runny nose, sore throat or diarrhea. These   symptoms are usually mild and begin gradually. Some people become infected but dont   develop any symptoms and don't feel unwell. Most people (about 80%) recover from the   disease without needing special treatment. Around 1 out of every 6 people who gets COVID-19   becomes seriously ill and develops difficulty breathing. Older people, and those with underlying   medical problems like high blood pressure, heart problems or diabetes, are more likely to   develop serious illness. Pe

In [35]:
final_response_sources = set()

for reranked_doc in reranked_docs:
  final_response_sources.add(reranked_doc.metadata["dataset"])

In [36]:
final_response_sources

{'COUGH_FAQ', 'COVID19_FAQ'}

### Post Retrieval Optimisation Pipeline

In [37]:
def post_retrieval_optimisation(
    query: str,
    retrieved_docs_with_scores: List[Tuple[Document, float]],
    rerank_top_k: int = 5,
) -> List[str]:
    """
    Post-retrieval optimisation:
    1. Cross-encoder reranking
    2. Prompt compression
    """

    # Strip FAISS scores
    retrieved_docs = [doc for doc, _ in retrieved_docs_with_scores]

    # Reranking
    reranker = CrossEncoderReranker(top_k=rerank_top_k)
    reranked_docs_with_scores = reranker.rerank(
        query=query,
        documents=retrieved_docs,
    )

    reranked_docs = [doc for doc, _ in reranked_docs_with_scores]

    # Prompt Compression
    compressor = PromptCompressor()
    compressed_context = compressor.compress(
        query=query,
        documents=reranked_docs,
    )

    return compressed_context
